# MappedDataSet: a shared, virtual dataset

`datasets.artifact.DataSet` builds a training set by concatenating every source's
`tokens.bin` into a new `train.bin`/`valid.bin`. `MappedDataSet` is the alternative: it
owns no files of its own at all and reads straight out of each source's own `tokens.bin`
through `mappeddatasets.tokenstream.TokenStream` -- one numpy memmap per source, addressed by a
single global index, and it refuses any window that would cross from one source into the
next rather than silently mixing them.

It's also shared, not run-scoped: its identity is a digest over the tokenizer and source
lists, so two notebooks asking for the same tokenizer and sources land on the same folder
and reuse each other's work. And it's *done* exactly when its sources are -- there is no
job output of its own to wait on.

This walks build, bind, read -- against a local folder, no Modal involved. See
[tokenizer_demo.ipynb](tokenizer_demo.ipynb) for the Tokenizer walkthrough this builds on.

In [ ]:
import logging
from pathlib import Path
from types import SimpleNamespace

from dag import resolve as dag_resolve
from datasets.artifact import DataSet
from mappeddatasets.artifact import MappedDataSet
from sources.artifact import Source
from tokenizers.bpe import Tokenizer

# the same throwaway volume stand-in tokenizer_demo.ipynb uses
ROOT = Path(".scratch/demo-volume").resolve()
ROOT.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
worker = SimpleNamespace(log=logging.getLogger("demo"))

ROOT

## Two sources, one tokenizer

In [ ]:
romeojuliet = Source(
    name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt"
)
mobydick = Source(
    name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt"
)

tokenizer = Tokenizer(
    vocab_size=1000, special_tokens=("<|endoftext|>",), sources=(romeojuliet, mobydick)
)

mapped = MappedDataSet.from_sources(
    tokenizer=tokenizer, train_sources=[romeojuliet, mobydick], valid_sources=[romeojuliet]
)

print(mapped.uid)            # no run_id -- identity is the tokenizer + these sources
print(mapped.artifact_path)  # mappeddatasets/mapped-<digest>, a shared folder
print(mapped.files)          # {} -- nothing of its own to write

## Declare and build

In [16]:
declaration = dag_resolve.Declaration(mapped, ROOT).check()
declaration  # reads the disk, writes nothing

run - under /Users/oguz/Projects/launchpad/.scratch/demo-volume
  done       sources/romeojuliet
  done       sources/mobydick
  done       tokenizers/bpe-1000-5332abba6c
  done       tokenizers/bpe-1000-5332abba6c/bin/romeojuliet
  done       tokenizers/bpe-1000-5332abba6c/bin/mobydick
  done       datasets/mapped-5ac8d79f02

6 done
ok -- 0 to declare

In [17]:
declaration.write()  # one manifest.json per artifact, dependencies first

for job in dag_resolve.job_list(dag_resolve.resolve(mapped)):
    if dag_resolve.status(job.artifact, ROOT) == "done":
        print(f"already done: {job.artifact.artifact_path}")
        continue
    job.run(ROOT, worker)

already done: sources/romeojuliet
already done: sources/mobydick
already done: tokenizers/bpe-1000-5332abba6c
already done: tokenizers/bpe-1000-5332abba6c/bin/romeojuliet
already done: tokenizers/bpe-1000-5332abba6c/bin/mobydick
already done: datasets/mapped-5ac8d79f02


## Bind and read

`bind` returns a *new* MappedDataSet with `train_tokens`/`valid_tokens` attached -- built
directly from `train_set`/`valid_set`, never by reading anything back from its own folder
(there's nothing there to read). The object you called `bind` on stays exactly as unbound
as it was, so the convention is to reassign: `mapped = mapped.bind(ROOT)`.

In [11]:
mapped = mapped.bind(ROOT)  # a new, bound MappedDataSet -- reassign, don't discard
print(f"{len(mapped.train_tokens)} train tokens across {len(mapped.train_set)} source(s)")

bound_tokenizer = tokenizer.bind(ROOT)
print(repr(bound_tokenizer.decode(mapped.train_tokens[:20])))  # a zero-copy memmap slice

564664 train tokens across 2 source(s)
'The Project Gutenberg eBook of Romeo and Juliet\n    \nThis eB'


In [13]:
mapped.train_tokens[4:20]

memmap([ 66, 477, 280, 615, 963, 285, 675, 393, 964,  10, 912,  10,  84,
        678, 318,  66], dtype=uint16)

## Sources never mix

`romeojuliet` and `mobydick` are two unrelated books -- a training window must never span
where one ends and the other begins. `TokenStream.locate` finds that boundary; asking for a
window that crosses it raises, rather than quietly handing back a page that is half one book
and half the other.

In [ ]:
from tokenizers.bpe import TokenizedSource

romeo_tokens = TokenizedSource(tokenizer, romeojuliet).paths(ROOT)["tokens"].stat().st_size // 2
print(mapped.train_tokens.locate(romeo_tokens))  # (1, 0) -- the first index of mobydick

try:
    mapped.train_tokens[romeo_tokens - 1 : romeo_tokens + 1]
except ValueError as err:
    print("refused ->", err)

## No duplication

Building the equivalent `DataSet` writes a fresh copy of every token into its own
`train.bin`. `MappedDataSet` writes nothing at all -- its folder holds only the manifest
`Declaration.write` puts there, and it reads `done` the moment its sources do.

In [ ]:
physical = DataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[romeojuliet, mobydick], valid_sources=[romeojuliet],
)
dag_resolve.Declaration(physical, ROOT).write()
for job in dag_resolve.job_list(dag_resolve.resolve(physical)):
    if dag_resolve.status(job.artifact, ROOT) == "done":
        continue
    job.run(ROOT, worker)

physical_bytes = physical.paths(ROOT)["training set"].stat().st_size
mapped_folder_contents = sorted(p.name for p in (ROOT / mapped.artifact_path).iterdir())
print(f"DataSet copies {physical_bytes:,} bytes into train.bin")
print(f"MappedDataSet's folder holds only: {mapped_folder_contents}")

Same tokens, not just the same size -- reading `MappedDataSet`'s view source by source and
concatenating should reproduce `DataSet`'s `train.bin` exactly, byte for byte.

In [ ]:
import numpy as np
from array import array

physical_bytes = physical.paths(ROOT)["training set"].read_bytes()

offsets = [0]
for ts in mapped.train_set:
    offsets.append(offsets[-1] + ts.paths(ROOT)["tokens"].stat().st_size // 2)

stitched = np.concatenate(
    [mapped.train_tokens[offsets[i] : offsets[i + 1]] for i in range(len(mapped.train_set))]
)
mapped_bytes = array("H", stitched.tolist()).tobytes()

assert mapped_bytes == physical_bytes
print(f"identical: {len(physical_bytes):,} bytes match exactly between DataSet and MappedDataSet")

In [ ]:
# Cleanup, if you want the demo volume gone:
# import shutil; shutil.rmtree(ROOT)